### Einfaches Model-View-Controller Pattern
Wir nutzen Vererbung, um ein einfaches MVC-Pattern zu Implementieren.  
Die Klasse `Observable` enthält Code, die jede unserer Game-Klassen haben soll und
die Klasse `BaseView` enthält Code, die jede unserer View-Klassen haben soll.

- Die Klasse `Observable` ermöglicht das Registrieren von Callbacks mit `register_callback(fun)` und das Aufrufen der registrierten Callbacks mit `_notify(event, data)`.
- Die Klasse `BaseView` hat Attribute `mcanvas` (MultiCanvas) und `canvas` (Top Layer der Multicanvas), sowie eine Methode `_ipython_display`, die aufgerufen wird, wenn die Klasse im Notebook dargestellt werden soll.  
  Zudem wird bei der Initialisierung `BaseView(game)` die Methode `update` als Callback bei
  der Game-Instanz `game` registriert

In [ ]:
import widget_helpers as W
from IPython.display import display
from functools import wraps


def notify(f):
    '''f(self, ...) ist eine Methode
       notify(f)(self, ...)
       - fuehrt data = f(self, ...) aus
       - ruft self._notify(f.__name__, data) auf
    '''
    @wraps(f)  # korrigiert u.a. f.__name__
    def wrapper(self, *args, **kwargs):
        data = f(self, *args, **kwargs)
        self._notify(f.__name__, data)
    return wrapper


class Observable:
    def __init__(self):
        self.callbacks = []

    def register_callback(self, callback):
        self.callbacks.append(callback)

    def _notify(self, event, data=None):
        for f in self.callbacks:
            f(event, data)


class Game(Observable):
    def __init__(self):
        super().__init__()
        self.player_pos = (50, 50)

    @notify
    def new_game(self):
        self.player_pos = (50, 50)

    @notify
    def move(self,  dx, dy):
        x, y = self.player_pos
        self.player_pos = (x + dx, y + dy)


class BaseView:
    def __init__(self, game, width=100, height=100, nlayers=1):
        self.game = game
        self.mcanvas = W.get_mcanvas(nlayers, width=width, height=height)
        self.canvas = self.mcanvas[-1]

        self.game.register_callback(self.update)

    def display(self):
        display(self.mcanvas)
        self.mcanvas.focus()

    def update(self, event, data):
        raise NotImplementedError

    def _ipython_display_(self):
        self.display()


class View(BaseView):
    def __init__(self, game):
        super().__init__(game)
        self.draw_player()

    def draw_player(self):
        self.canvas.clear()
        self.canvas.fill_circle(*self.game.player_pos, 5)

    def update(self, event, data):
        self.draw_player()


class Controller:
    def __init__(self, game, view, callbacks):
        self.game = game
        self.view = view
        self.callbacks = callbacks
        self.view.mcanvas.on_key_down(self.on_key_down)

    def on_key_down(self, key, *flags):
        if key in self.callbacks:
            self.callbacks[key]()

    def _ipython_display_(self):
        self.view.display()

In [ ]:
game = Game()
callbacks = {'n': game.new_game,
             'ArrowUp': lambda: game.move(0, -10),
             'ArrowDown': lambda: game.move(0, 10),
             'ArrowRight': lambda: game.move(10, 0),
             'ArrowLeft': lambda: game.move(-10, 0),
             }
view = View(game)
controller = Controller(game, view, callbacks)
controller